In [ ]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Cannot find repo root containing data/ and notebooks/")

REPO_ROOT = find_repo_root(Path.cwd())
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

customers = pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW_DATA_DIR / "olist_geolocation_dataset.csv")
interface = pd.read_csv(RAW_DATA_DIR / "olist_interface_score_dataset.csv")
order_items = pd.read_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv")
payment = pd.read_csv(RAW_DATA_DIR / "olist_order_payments_dataset.csv")
tran_reviews = pd.read_csv(RAW_DATA_DIR / "olist_order_reviews_dataset_translated_nodup.csv")
dataset = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv")
products = pd.read_csv(RAW_DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv")
category = pd.read_csv(RAW_DATA_DIR / "product_category_name_translation.csv")

Part 1: Find out the zip_code_prefix's geolocation

In [ ]:
# delete column 'city'
zip_location_nocity = geolocation.drop('geolocation_city', axis=1)
display(zip_location_nocity)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_state
0,1037,-23.545621,-46.639292,SP
1,1046,-23.546081,-46.644820,SP
2,1046,-23.546129,-46.642951,SP
3,1041,-23.544392,-46.639499,SP
4,1035,-23.541578,-46.641607,SP
...,...,...,...,...
1000158,99950,-28.068639,-52.010705,RS
1000159,99900,-27.877125,-52.224882,RS
1000160,99950,-28.071855,-52.014716,RS
1000161,99980,-28.388932,-51.846871,RS


In [ ]:
# group by zip_code_prefix (Since a zip_code_prefix has more than one geolocation)
zip_location_group = zip_location_nocity.groupby('geolocation_zip_code_prefix').agg({'geolocation_lat':'mean',
 'geolocation_lng':'mean', 'geolocation_state':lambda x: x.iloc[0]})
display(zip_location_group)

,geolocation_lat,geolocation_lng,geolocation_state
geolocation_zip_code_prefix,,,
1001,-23.550190,-46.634024,SP
1002,-23.548146,-46.634979,SP
1003,-23.548994,-46.635731,SP
1004,-23.549799,-46.634757,SP
1005,-23.549456,-46.636733,SP
...,...,...,...
99960,-27.953722,-52.025511,RS
99965,-28.183372,-52.039850,RS
99970,-28.343766,-51.874689,RS


In [ ]:
# for test
display(zip_location_group.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19015 entries, 1001 to 99990
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   geolocation_lat    19015 non-null  float64
 1   geolocation_lng    19015 non-null  float64
 2   geolocation_state  19015 non-null  object 
dtypes: float64(2), object(1)
memory usage: 594.2+ KB


None

Part 2: Find out the seller's geolocation

In [ ]:
# combine tables 'sellers' + 'zip_location_group'
seller_geolocation = pd.merge(sellers, zip_location_group, how = 'left', left_on = 'seller_zip_code_prefix', right_on = 'geolocation_zip_code_prefix')
display(seller_geolocation)

,seller_id,seller_zip_code_prefix,seller_city,seller_state,geolocation_lat,geolocation_lng,geolocation_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,-22.893848,-47.061337,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,-22.383437,-46.947927,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,-22.909572,-43.177703,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,-23.657242,-46.612831,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,-22.964803,-46.534419,SP
...,...,...,...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR,-23.448041,-51.869960,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC,-27.656421,-48.665840,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP,-23.657851,-46.676925,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS,-31.751072,-52.323202,RS


In [ ]:
# delete column 'geolocation_state'
# seller_geolocation = seller_geolocation.drop(['geolocation_state'], axis=1)
seller_geolocation = seller_geolocation.drop(['seller_city'], axis=1)

In [ ]:
# rename columns
seller_geolocation = seller_geolocation.rename(columns={'geolocation_lat':'sell_lat', 'geolocation_lng':'sell_lng'})

In [ ]:
# fill in some lat & lng which are null value
seller_geolocation[seller_geolocation.sell_lat.isnull()]

,seller_id,seller_zip_code_prefix,seller_state,sell_lat,sell_lng,geolocation_state
473,5962468f885ea01a1b6a97a218797b0a,82040,PR,NaN,NaN,NaN
791,2aafae69bf4c41fbd94053d9413e87ee,91901,RS,NaN,NaN,NaN
1672,2a50b7ee5aebecc6fd0ff9784a4747d6,72580,DF,NaN,NaN,NaN
1931,2e90cb1677d35cfe24eef47d441b7c87,2285,SP,NaN,NaN,NaN
2182,0b3f27369a4d8df98f7eb91077e438ac,7412,SP,NaN,NaN,NaN
2986,42bde9fef835393bb8a8849cb6b7f245,71551,DF,NaN,NaN,NaN
3028,870d0118f7a9d85960f29ad89d5d989a,37708,MG,NaN,NaN,NaN


In [ ]:
# make the index of null row in sell_lat + sell_lng
index_list = seller_geolocation[seller_geolocation.sell_lat.isnull()].index
display(index_list)

Int64Index([473, 791, 1672, 1931, 2182, 2986, 3028], dtype='int64')

In [ ]:
# for index in index_list:
#   seller_geolocation.at[index, 'sell_lat'] = seller_geolocation.at[index, 'sell_lat']
#   seller_geolocation.at[index, 'sell_lng'] = seller_geolocation.at[index, 'sell_lng']

for index in index_list:
   seller_state_1 = seller_geolocation.seller_state[index]
   print(seller_state_1)
   seller_geolocation.sell_lat[index] = seller_geolocation.loc[seller_geolocation['seller_state'] == seller_state_1, 'sell_lat'].iloc[0]
   seller_geolocation.sell_lng[index] = seller_geolocation.loc[seller_geolocation['seller_state'] == seller_state_1, 'sell_lng'].iloc[0]


In [64]:
seller_geolocation.info() # show the empty lat & lng have been filled

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3095 entries, 0 to 3094
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   seller_id               3095 non-null   object 
 1   seller_zip_code_prefix  3095 non-null   int64  
 2   seller_state            3095 non-null   object 
 3   sell_lat                3095 non-null   float64
 4   sell_lng                3095 non-null   float64
 5   geolocation_state       3088 non-null   object 
dtypes: float64(2), int64(1), object(3)
memory usage: 233.8+ KB


In [ ]:
# to mark each order's seller geolocation (combine tables 'order_items' + 'seller_geolocation')

order_seller_geolocation_notclean = pd.merge(order_items, seller_geolocation, how = 'left', on = 'seller_id')
display(order_seller_geolocation_notclean)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_state,sell_lat,sell_lng,geolocation_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,SP,-22.496953,-44.127492,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,SP,-23.565096,-46.518565,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,MG,-22.262584,-46.171124,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,SP,-20.553624,-47.387359,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,PR,-22.929384,-53.135873,PR
...,...,...,...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,88303,SC,-26.912574,-48.673980,SC
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,1206,SP,-23.535864,-46.642819,SP
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,80610,PR,-25.469955,-49.289821,PR
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,4733,SP,-23.635530,-46.694031,SP


In [70]:
# delete unwanted columns
order_seller_geolocation = order_seller_geolocation_notclean.drop(['order_item_id', 'price', 'shipping_limit_date', 'freight_value', 'geolocation_state'], axis=1)


In [71]:
order_seller_geolocation.head()

,order_id,product_id,seller_id,seller_zip_code_prefix,seller_state,sell_lat,sell_lng
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,27277,SP,-22.496953,-44.127492
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,3471,SP,-23.565096,-46.518565
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,37564,MG,-22.262584,-46.171124
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,14403,SP,-20.553624,-47.387359
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,87900,PR,-22.929384,-53.135873


In [74]:
order_seller_geolocation.info() # perfect!

<class 'pandas.core.frame.DataFrame'>
Int64Index: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                112650 non-null  object 
 1   product_id              112650 non-null  object 
 2   seller_id               112650 non-null  object 
 3   seller_zip_code_prefix  112650 non-null  int64  
 4   seller_state            112650 non-null  object 
 5   sell_lat                112650 non-null  float64
 6   sell_lng                112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.9+ MB


In [75]:
# export file
order_seller_geolocation.to_csv(RAW_DATA_DIR / "order_geolocation.csv", index=False)

In [ ]:
# the seller_location's table basically done, next should make the customer_location's table

Part 3: Find out the customer's geolocation

In [ ]:
# combine tables 'customers' + 'zip_location_group'
customer_location_notclean = pd.merge(customers, zip_location_group, how='left', left_on='customer_zip_code_prefix', right_on = 'geolocation_zip_code_prefix')

In [ ]:
customer_location_notclean.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_lat,geolocation_lng,geolocation_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,-20.498489,-47.396929,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,-23.727992,-46.542848,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,-23.531642,-46.656289,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,-23.499702,-46.185233,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,-22.975100,-47.142925,SP


In [ ]:
# delete unwanted columns
customer_location = customer_location_notclean.drop(['customer_unique_id', 'customer_city', 'geolocation_state'], axis=1)
display(customer_location)

,customer_id,customer_zip_code_prefix,customer_state,geolocation_lat,geolocation_lng
0,06b8999e2fba1a1fbc88172c00ba8bc7,14409,SP,-20.498489,-47.396929
1,18955e83d337fd6b2def6b18a428ac77,9790,SP,-23.727992,-46.542848
2,4e7b3e00288586ebd08712fdd0374a03,1151,SP,-23.531642,-46.656289
3,b2b6027bc5c5109e529d4dc6358b12c3,8775,SP,-23.499702,-46.185233
4,4f2d8ab171c80ec8364f7c12e35b23ad,13056,SP,-22.975100,-47.142925
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,3937,SP,-23.586003,-46.499638
99437,e7b71a9017aa05c9a7fd292d714858e8,6764,SP,-23.615830,-46.768533
99438,5e28dfe12db7fb50a4b2f691faecea5e,60115,CE,-3.734569,-38.510534
99439,56b18e2166679b8a959d72dd06da27f9,92120,RS,-29.949839,-51.168494


In [ ]:
# to link up table 'order_item', must combine two tables 'customer_location' + 'orders_dataset'
order_cust_location_notclean = pd.merge(dataset, customer_location, how='left', on='customer_id')
order_cust_location_notclean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,customer_state,geolocation_lat,geolocation_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,3149,SP,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,47813,BA,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,75265,GO,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,59296,RN,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,9195,SP,-23.676370,-46.514627


In [ ]:
order_cust_location_notclean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,customer_state,cust_lat,cust_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,3149,SP,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,47813,BA,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,75265,GO,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,59296,RN,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,9195,SP,-23.676370,-46.514627


In [ ]:
# delete unwanted columns
order_cust_location = order_cust_location_notclean.drop(['order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'], axis=1)
order_cust_location.head()

,order_id,customer_id,customer_zip_code_prefix,customer_state,cust_lat,cust_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,3149,SP,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,47813,BA,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,75265,GO,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,59296,RN,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,9195,SP,-23.676370,-46.514627


In [ ]:
order_cust_location.info() # shown some lat & lng are missing, fixing needed

<class 'pandas.core.frame.DataFrame'>
Int64Index: 99441 entries, 0 to 99440
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_id                  99441 non-null  object 
 1   customer_id               99441 non-null  object 
 2   customer_zip_code_prefix  99441 non-null  int64  
 3   customer_state            99441 non-null  object 
 4   cust_lat                  99163 non-null  float64
 5   cust_lng                  99163 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 5.3+ MB


In [ ]:
# insert the value to lat & lng, which are null
# make the index of null row in cust_lat + cust_lng
index_list_2 = order_cust_location[order_cust_location.cust_lat.isnull()].index
display(index_list_2)

Int64Index([  445,   610,   681,   926,  1091,  1202,  1343,  2004,  2344,
             2495,
            ...
            92991, 94493, 95336, 95628, 95962, 97935, 98886, 99062, 99162,
            99217],
           dtype='int64', length=278)

In [ ]:
for index_2 in index_list_2:
   cust_state_1 = order_cust_location.customer_state[index_2]
   print(cust_state_1)
   order_cust_location.cust_lat[index_2] = order_cust_location.loc[order_cust_location['customer_state'] == cust_state_1, 'cust_lat'].iloc[0]
   order_cust_location.cust_lng[index_2] = order_cust_location.loc[order_cust_location['customer_state'] == cust_state_1, 'cust_lng'].iloc[0]

In [ ]:
order_cust_location.info() # fixed

<class 'pandas.core.frame.DataFrame'>
Int64Index: 99441 entries, 0 to 99440
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_id                  99441 non-null  object 
 1   customer_id               99441 non-null  object 
 2   customer_zip_code_prefix  99441 non-null  int64  
 3   customer_state            99441 non-null  object 
 4   cust_lat                  99441 non-null  float64
 5   cust_lng                  99441 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 7.3+ MB


In [ ]:
order_cust_location.head()

,order_id,customer_id,customer_zip_code_prefix,customer_state,cust_lat,cust_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,3149,SP,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,47813,BA,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,75265,GO,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,59296,RN,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,9195,SP,-23.676370,-46.514627


In [ ]:
# combine tables order_items & order_cust_location
order_cust_location_2 = pd.merge(order_items, order_cust_location, how='left', on='order_id')
order_cust_location_2.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,customer_zip_code_prefix,customer_state,cust_lat,cust_lng
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,28013,RJ,-21.762775,-41.309633
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,15775,SP,-20.220527,-50.903424
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,35661,MG,-19.870305,-44.593326
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,12952,SP,-23.089925,-46.611654
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,13226,SP,-23.243402,-46.827614


In [ ]:
# delete unwanted columns
order_cust_location_2 = order_cust_location_2.drop(['order_item_id', 'seller_id', 'price', 'shipping_limit_date', 'freight_value'], axis=1)
order_cust_location_2.head()

,order_id,product_id,customer_id,customer_zip_code_prefix,customer_state,cust_lat,cust_lng
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,3ce436f183e68e07877b285a838db11a,28013,RJ,-21.762775,-41.309633
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,f6dd3ec061db4e3987629fe6b26e5cce,15775,SP,-20.220527,-50.903424
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,6489ae5e4333f3693df5ad4372dab6d3,35661,MG,-19.870305,-44.593326
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,d4eb9395c8c0431ee92fce09860c5a06,12952,SP,-23.089925,-46.611654
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,58dbd0b2d70206bf40e62cd34e84d795,13226,SP,-23.243402,-46.827614


In [ ]:
order_cust_location_2.info() # perfect!

<class 'pandas.core.frame.DataFrame'>
Int64Index: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  112650 non-null  object 
 1   product_id                112650 non-null  object 
 2   customer_id               112650 non-null  object 
 3   customer_zip_code_prefix  112650 non-null  int64  
 4   customer_state            112650 non-null  object 
 5   cust_lat                  112650 non-null  float64
 6   cust_lng                  112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.9+ MB


In [ ]:
# export file
order_cust_location_2.to_csv(PROCESSED_DATA_DIR / "order_geolocation_2.csv", index=False)